# EP.2 归一化：统计轴、残差位置与 QK 尺度

配套文章：[一文搞懂归一化技术](https://momoyeyu.github.io/posts/llm-normalization/)。上一章：[01_moe.ipynb](01_moe.ipynb)；下一章：[03_position.ipynb](03_position.ipynb)。本章沿用 [EP.0](00_transformer.ipynb) 定义的 `set_seed` / `show`，从统计量的“沿哪些轴计算”出发，实现 BatchNorm、LayerNorm、RMSNorm，并比较归一化放在残差前后时的结构差异。

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "slm").is_dir():
    root = next((parent for parent in root.parents if (parent / "slm").is_dir()), None)
if root is None:
    raise RuntimeError("请从仓库根目录或 notebooks 目录运行")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from slm import set_seed, show

torch.set_num_threads(1)
set_seed(0)
show("Python", sys.version.split()[0])
show("PyTorch", torch.__version__)
show("device", "cpu")
import time

Python                       3.12.12
PyTorch                      2.14.1
device                       cpu


## 1. 残差流为何会漂移？

若每层叠加独立、零均值、方差 0.01 的扰动，则 $\mathrm{Var}(x_L)=1+0.01L$，标准差是其平方根。

In [2]:
import math


import torch


set_seed(0)

x = torch.randn(8, 128, 512)               # [N, L, C]
stds = [x.std().item()]
for _ in range(64):
    x = x + torch.randn(8, 128, 512) * 0.1   # 模拟各子层的输出
    stds.append(x.std().item())

for depth in [0, 16, 32, 64]:
    expected = math.sqrt(1 + depth * 0.1**2)
    show(f"depth {depth:<3} std", f"observed {stds[depth]:.3f} / expected {expected:.3f}")

depth 0   std                observed 1.000 / expected 1.000
depth 16  std                observed 1.079 / expected 1.077
depth 32  std                observed 1.150 / expected 1.149
depth 64  std                observed 1.283 / expected 1.281


观测值应接近独立噪声公式；真实 learned residual 往往相关且非零均值，所以这不是深网残差尺度的普遍定理。

## 2. BatchNorm：跨 N、L 的通道统计

对 `[N,L,C]`，每个通道在 `(N,L)` 上统计。训练方差用于当前输出时取 biased 估计；running variance 按 PyTorch 惯例更新 unbiased 估计。

In [3]:
class BatchNorm(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1) -> None:
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, L, C]
        if self.training:
            mean = x.mean(dim=(0, 1))                    # [C]
            var = x.var(dim=(0, 1), unbiased=False)      # [C]
            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                # PyTorch 惯例：running_var 用无偏估计
                self.running_var.mul_(1 - self.momentum).add_(
                    self.momentum * x.var(dim=(0, 1), unbiased=True)
                )
        else:
            mean, var = self.running_mean, self.running_var
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

In [4]:
import torch


set_seed(0)
bn = BatchNorm(num_features=8)
x = torch.randn(4, 16, 8) + 2  # [N, L, C]

y = bn(x)
show("train out mean max", y.mean(dim=(0, 1)).abs().max().item())
show("train out var", y.var(dim=(0, 1), unbiased=False).mean().item())
show("running_mean[0]", f"{bn.running_mean[0].item():.2f} (momentum 0.1, target 2)")

# 推理切到 running 统计量，不再用当前 batch 的均值方差
bn.eval()
manual = (x - bn.running_mean) / torch.sqrt(bn.running_var + bn.eps) * bn.weight + bn.bias
show("eval uses running stats", torch.allclose(bn(x), manual, atol=1e-6))

# padding 污染：同一批数据补长一倍后，batch 统计量明显漂移
x_padded = torch.cat([x, torch.zeros(4, 16, 8)], dim=1)
show("batch mean padded", f"{x.mean(dim=(0, 1))[0]:.3f} -> {x_padded.mean(dim=(0, 1))[0]:.3f}")
show("batch var padded", f"{x.var(dim=(0, 1), unbiased=False)[0]:.3f} -> {x_padded.var(dim=(0, 1), unbiased=False)[0]:.3f}")

train out mean max           2.123415470123291e-07
train out var                0.9999896287918091
running_mean[0]              0.19 (momentum 0.1, target 2)
eval uses running stats      True
batch mean padded            1.930 -> 0.965
batch var padded             0.697 -> 1.279


train/eval 使用不同统计量，padding 零值会污染 batch 统计。本教学实现只处理 `[N,L,C]`，训练时 unbiased running variance 至少需要两个观测。

## 3. LayerNorm：每个 token 自己归一化

LayerNorm 只沿最后通道 C 统计，因此其他 token 是否 padding 不改变当前 token。

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, L, C]
        mean = x.mean(dim=-1, keepdim=True)                       # [N, L, 1]
        var = x.var(dim=-1, unbiased=False, keepdim=True)         # [N, L, 1]
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

In [6]:
import torch
import torch.nn as nn


set_seed(0)
ln = LayerNorm(768)
x = torch.randn(4, 16, 768)  # [N, L, C]

show("matches nn.LayerNorm", torch.allclose(ln(x), nn.LayerNorm(768)(x), atol=1e-6))
show("shift invariant", torch.allclose(ln(x + 5), ln(x), atol=1e-5))
show("scale invariant", torch.allclose(ln(x * 3), ln(x), atol=1e-5))

# 统计只看单个 token：其他位置补 padding 不影响本 token 的输出
x_pad = x.clone()
x_pad[:, 8:] = 0
show("immune to padding", torch.allclose(ln(x_pad)[:, :8], ln(x)[:, :8]))

matches nn.LayerNorm         True
shift invariant              True
scale invariant              True
immune to padding            True


平移会同时平移均值，因此理想算术下平移不变；正比例缩放只在忽略 `eps` 时严格不变。负比例会翻转标准化部分的符号。

## 4. RMSNorm：只保留均方根缩放

RMSNorm 不减均值，也没有 beta，只学习逐通道 gamma；统计量用 fp32 计算后转回输入精度。

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, L, C]
        dtype = x.dtype
        x = x.float()                                        # 统计量用 fp32
        x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)  # [N,L,C] × [N,L,1]
        return self.weight * x.to(dtype)                     # 转回原精度再乘权重（LLaMA 顺序）

In [8]:
import time


import torch
import torch.nn as nn


set_seed(0)
x = torch.randn(4, 16, 768)
reference = (
    nn.RMSNorm(768, eps=1e-6)(x)
    if hasattr(nn, "RMSNorm")
    else x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)
)
show("matches reference", torch.allclose(RMSNorm(768, eps=1e-6)(x), reference, atol=1e-6))
show("parameters", [n for n, _ in RMSNorm(8).named_parameters()])

if hasattr(nn, "RMSNorm"):
    x_b = torch.randn(4096, 768)
    modules = [
        ("LayerNorm", nn.LayerNorm(768)),
        ("taught RMSNorm", RMSNorm(768)),
        ("native RMSNorm", nn.RMSNorm(768)),
    ]
    for name, module in modules:
        module(x_b)
        t0 = time.perf_counter()
        for _ in range(300):
            module(x_b)
        show(name, f"{(time.perf_counter() - t0) * 1000:.0f} ms / 300 iters")

matches reference            True
parameters                   ['weight']


LayerNorm                    511 ms / 300 iters


taught RMSNorm               621 ms / 300 iters


native RMSNorm               613 ms / 300 iters


对拍验证公式。计时仅是当前 CPU、eager、forward 的局部测量；不能据此声称 RMSNorm 在所有设备、编译器或训练负载上更快。

## 5. Pre-Norm 与 Post-Norm

Post：`norm(x + F(x))`；Pre：`x + F(norm(x))`。复用刚实现的 RMSNorm，避免把核心机制藏在原生层中。

In [9]:
class PostNormBlock(nn.Module):
    def __init__(self, d_model: int, sublayer: nn.Module) -> None:
        super().__init__()
        self.sublayer = sublayer
        self.norm = RMSNorm(d_model, eps=torch.finfo(torch.float32).eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, L, C]
        return self.norm(x + self.sublayer(x))   # 残差之后归一化

In [10]:
class PreNormBlock(nn.Module):
    def __init__(self, d_model: int, sublayer: nn.Module) -> None:
        super().__init__()
        self.sublayer = sublayer
        self.norm = RMSNorm(d_model, eps=torch.finfo(torch.float32).eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, L, C]
        return x + self.sublayer(self.norm(x))   # 归一化后进子层

In [11]:
import torch
import torch.nn as nn


D, DEPTH, STEPS, LR = 32, 32, 30, 0.5


def make_stack(block_cls):
    return nn.Sequential(*[
        block_cls(D, nn.Sequential(nn.Linear(D, D * 2), nn.GELU(), nn.Linear(D * 2, D)))
        for _ in range(DEPTH)
    ])


# Post-Norm normalizes each output; Pre-Norm leaves the residual stream unnormalized.
set_seed(0)
x = torch.randn(4, 16, D)
for name, block_cls in [("post", PostNormBlock), ("pre", PreNormBlock)]:
    stack = make_stack(block_cls)
    with torch.no_grad():
        h = x
        for blk in stack:
            h = blk(h)
    show(f"{name}-norm stream std", f"{x.std():.2f} -> {h.std():.2f} after {DEPTH} blocks")

# This fixed-learning-rate run illustrates one setting, not a universal stability result.
x, target = torch.randn(8, 16, D), torch.randn(8, 16, D)
for name, block_cls in [("post", PostNormBlock), ("pre", PreNormBlock)]:
    set_seed(0)
    stack = make_stack(block_cls)
    opt = torch.optim.SGD(stack.parameters(), lr=LR)
    losses = []
    for _ in range(STEPS):
        loss = nn.functional.mse_loss(stack(x), target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    show(f"{name}-norm SGD lr={LR}", f"{losses[0]:.2f} -> {losses[9]:.2f} -> {losses[-1]:.4f}")

post-norm stream std         1.02 -> 1.00 after 32 blocks
pre-norm stream std          1.02 -> 1.58 after 32 blocks


post-norm SGD lr=0.5         1.94 -> 0.90 -> 0.7324


pre-norm SGD lr=0.5          3.61 -> 1.30 -> 0.9009


EP.0 的 Transformer 使用 LayerNorm，而此实验使用 RMSNorm；归一化“放在哪里”的思想与具体 norm 类型独立。真实 Pre-Norm 堆栈通常还需最终 norm。单个 seed、深度和学习率不能推出哪种结构总是更稳定。

## 6. QK-Norm：直接控制 attention logits

在 Q/K 投影后逐 head 归一化，再做点积。这里不提前依赖下一章 RoPE。

In [12]:
import math


import torch


set_seed(0)
B, L, H, D = 2, 16, 4, 32
d_model = H * D

q_proj = torch.nn.Linear(d_model, d_model, bias=False)
k_proj = torch.nn.Linear(d_model, d_model, bias=False)
q_norm = RMSNorm(D)
k_norm = RMSNorm(D)

x = torch.randn(B, L, d_model)
with torch.no_grad():  # 放大投影权重模拟训练后期 logits 膨胀
    q_proj.weight *= 20
    k_proj.weight *= 20


def logits(norm: bool) -> torch.Tensor:
    q = q_proj(x).view(B, L, H, D).transpose(1, 2)  # [B, H, L, D]
    k = k_proj(x).view(B, L, H, D).transpose(1, 2)  # [B, H, L, D]
    if norm:
        q, k = q_norm(q), k_norm(k)
    return q @ k.transpose(-1, -2) / math.sqrt(D)  # [B, H, L, L]


raw, normed = logits(False), logits(True)
show("raw logits |max|/std", f"{raw.abs().max():.1f} / {raw.std():.1f}")
show("normalized logits |max|/std", f"{normed.abs().max():.2f} / {normed.std():.2f}")
show("initial gamma=1 bound", f"sqrt(d_head) = {math.sqrt(D):.1f}")

raw logits |max|/std         673.6 / 135.8
normalized logits |max|/std  3.43 / 1.02
initial gamma=1 bound        sqrt(d_head) = 5.7


gamma 初始为 1 时，每个向量模长约为 $\sqrt D$，缩放点积绝对值上界为 $\sqrt D$。gamma 学习后会改变这个界。下一章会把 QK-Norm 与 RoPE 真正串起来。